In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from collections import namedtuple
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

# Recursive structure for loading a block
SavedBlock = namedtuple('SavedBlock', [
    'data',          # pandas DataFrame containing all block data
    'model',         # trained XGBoost model
    'metadata',      # metadata dictionary
    'feature_names', # list of feature names (excluding target and frequency)
    'target_name',   # target column name
    'frequency_name' # frequency column name
])


class SpectralFeatureSelector:
    """
    Spectral feature selector using XGBoost and splitting into blocks.
    Supports saving quality blocks and reloading them.
    """

    def __init__(
            self,
            df_start,
            target_name,
            frequency_name,
            is_norm=False,
            block_size=20,
            num_blocks=None,
            best_r2_train=0.69,
            best_r2_test=0.39,
            best_features_selected=4,
            test_size_ml=0.1,
            random_state_tr=4,
            xgb_params=None,
            # New parameters for saving
            save_blocks=False,                    # Enable/disable saving
            save_dir=None,                        # Save directory (default ./saved_blocks)
            save_r2_train_threshold=None,         # R² train threshold to save (default = best_r2_train)
            save_r2_test_threshold=None):         # R² test threshold to save (default = best_r2_test)

        self.df_satart = df_start.copy()
        self.target_name = target_name
        self.frequency_name = frequency_name
        self.block_size = block_size
        self.num_blocks = num_blocks
        self.best_r2_train = best_r2_train
        self.best_r2_test = best_r2_test
        self.best_features_selected = best_features_selected
        self.test_size_ml = test_size_ml
        self.random_state_tr = random_state_tr
        self.is_norm = is_norm

        # Saving configurations
        self.save_blocks = save_blocks
        self.save_dir = save_dir if save_dir is not None else "./saved_blocks"
        self.save_r2_train_threshold = save_r2_train_threshold if save_r2_train_threshold is not None else best_r2_train
        self.save_r2_test_threshold = save_r2_test_threshold if save_r2_test_threshold is not None else best_r2_test

        # Default XGBoost hyperparameters
        default_xgb_params = {
            "n_estimators": 150,
            "learning_rate": 0.03,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
            "random_state": 4,
            "min_child_weight": 0.4,
            "n_jobs": -1
        }
        if xgb_params:
            default_xgb_params.update(xgb_params)
        self.xgb_params = default_xgb_params

        # Data normalization and preparation
        self.df, self.frequency, self.target = self.__normaling_data_frame()
        self.blocks = self.__split_columns_into_blocks()

    ####################################################################
    # Private Methods
    ####################################################################

    def __normaling_data_frame(self):
        """Normalize data if necessary"""
        if self.is_norm is False:
            scaler = MinMaxScaler(feature_range=(0.2, 1))
            self.df_satart[self.target_name] = scaler.fit_transform(
                self.df_satart[[self.target_name]])
            self.df_satart[self.frequency_name] = scaler.fit_transform(
                self.df_satart[[self.frequency_name]])

            target = self.df_satart[self.target_name]
            frequency = self.df_satart[self.frequency_name]

            df = self.df_satart.copy()
            df = df.drop([self.target_name, self.frequency_name],
                         axis=1, errors='ignore')
            return df, frequency, target
        elif self.is_norm is True:
            target = self.df_satart[self.target_name]
            frequency = self.df_satart[self.frequency_name]

            df = self.df_satart.copy()
            df = df.drop([self.target_name, self.frequency_name],
                         axis=1, errors='ignore')
            return df, frequency, target
        else:
            raise ValueError("is_norm must be True or False")

    def __split_columns_into_blocks(self):
        """Split feature columns into blocks of a specific size"""
        columns = list(self.df.columns)
        blocks = []

        for i in range(0, len(columns), self.block_size):
            block_cols = columns[i:i+self.block_size]
            block_df = self.df[block_cols].copy()
            block_df.insert(0, self.frequency_name, self.frequency.values)
            block_df[self.target_name] = self.target.values
            blocks.append(block_df)

        return blocks

    ####################################################################
    # Analyze a Block (Training and Evaluation)
    ####################################################################
    def analyze_block(self, df_selected, block_number):
        """
        Train the model on a block and return metrics, top features, and the model itself.
        """
        X = df_selected.drop(self.target_name, axis=1).values
        y = df_selected[self.target_name].values.reshape(-1, 1)

        scaler = StandardScaler()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size_ml, random_state=self.random_state_tr)

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = XGBRegressor(**self.xgb_params)
        model.fit(X_train, y_train.ravel())

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
        r2_train = r2_score(y_train, y_train_pred)
        r2_test = r2_score(y_test, y_test_pred)

        print("=" * 60)
        print(f"Block {block_number}")
        print(f"✅ RMSE on Test Set: {rmse_test:.4f}")
        print(f"✅ R² Score on Train Set: {r2_train:.4f}")
        print(f"✅ R² Score on Test Set: {r2_test:.4f}")
        print("=" * 60)

        cv = ShuffleSplit(
            n_splits=5, test_size=self.test_size_ml, random_state=self.random_state_tr)
        cv_scores = cross_val_score(model, X, y.ravel(), cv=cv, scoring='r2')

        print(f"✅ Cross-validation R² scores: {np.round(cv_scores, 4)}")
        print(
            f"✅ Mean CV R²: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

        importance = pd.DataFrame({
            "Feature": df_selected.drop(columns=self.target_name).columns,
            "Importance": model.feature_importances_
        })

        importance = importance.sort_values(
            by="Importance",
            ascending=False
        )

        best_features = importance["Feature"].tolist()

        print("\nBest features based on the XGBoost model itself:")
        print(best_features[:10])

        return r2_train, r2_test, best_features, model

    ####################################################################
    # Save Block (If enabled)
    ####################################################################
    def _save_block(self, block_idx, block_df, model, r2_train, r2_test, best_features):
        """
        Save data, model, and metadata of a block to a specified path.
        """
        if not self.save_blocks:
            return

        save_path = os.path.join(self.save_dir, f"block_{block_idx}")
        os.makedirs(save_path, exist_ok=True)

        # Save dataframe
        block_df.to_csv(os.path.join(save_path, "block_data.csv"), index=False)

        # Save model
        joblib.dump(model, os.path.join(save_path, "model.joblib"))

        # Save metadata (including all configurations and results)
        metadata = {
            "block_number": block_idx,
            "r2_train": float(r2_train),
            "r2_test": float(r2_test),
            "best_features_selected": self.best_features_selected,
            "best_features": best_features[:self.best_features_selected],
            "xgb_params": self.xgb_params,
            "block_size": self.block_size,
            "test_size_ml": self.test_size_ml,
            "random_state_tr": self.random_state_tr,
            "is_norm": self.is_norm,
            "save_r2_train_threshold": self.save_r2_train_threshold,
            "save_r2_test_threshold": self.save_r2_test_threshold,
            "target_name": self.target_name,
            "frequency_name": self.frequency_name
        }
        with open(os.path.join(save_path, "metadata.json"), "w") as f:
            json.dump(metadata, f, indent=4)

        print(f"✅ Block {block_idx} saved to {save_path}")

    ####################################################################
    # Static Method to Load a Saved Block
    ####################################################################
    @staticmethod
    def load_saved_block(block_path):
        """
        Load a saved block from a specified path.

        Parameters
        ----------
        block_path : str
            Path to the block folder (e.g., './saved_blocks/block_0')

        Returns
        -------
        SavedBlock
            A namedtuple containing:
                - data: pandas DataFrame
                - model: XGBRegressor
                - metadata: dict
                - feature_names: list
                - target_name: str
                - frequency_name: str
        """
        # Load metadata
        metadata_path = os.path.join(block_path, 'metadata.json')
        if not os.path.exists(metadata_path):
            raise FileNotFoundError(f"metadata.json not found in {block_path}")

        with open(metadata_path, 'r') as f:
            metadata = json.load(f)

        # Load model
        model_path = os.path.join(block_path, 'model.joblib')
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"model.joblib not found in {block_path}")

        model = joblib.load(model_path)

        # Load data
        data_path = os.path.join(block_path, 'block_data.csv')
        if not os.path.exists(data_path):
            raise FileNotFoundError(
                f"block_data.csv not found in {block_path}")

        data = pd.read_csv(data_path)

        # Extract names from metadata
        target_name = metadata.get('target_name')
        frequency_name = metadata.get('frequency_name')
        if target_name is None or frequency_name is None:
            raise KeyError(
                "target_name or frequency_name not found in metadata.")

        feature_names = [col for col in data.columns if col not in [
            target_name, frequency_name]]

        return SavedBlock(
            data=data,
            model=model,
            metadata=metadata,
            feature_names=feature_names,
            target_name=target_name,
            frequency_name=frequency_name
        )

    ####################################################################
    # Main Execution
    ####################################################################
    def run(self):
        """
        Execute the feature selection process on blocks and collect the best features.
        Also, if saving is enabled, qualified blocks are saved.
        """
        top_best_features = []

        if self.num_blocks is None:
            n_blocks = len(self.blocks)
        else:
            n_blocks = min(self.num_blocks, len(self.blocks))

        for i in range(n_blocks):
            r2_train, r2_test, best_features, model = self.analyze_block(
                self.blocks[i],
                i
            )

            # Feature selection condition (with primary thresholds)
            if (r2_train >= self.best_r2_train and r2_test > self.best_r2_test):
                top_best_features.extend(
                    best_features[:self.best_features_selected])

            # Saving condition (with specific save thresholds)
            if (self.save_blocks and
                r2_train >= self.save_r2_train_threshold and
                    r2_test >= self.save_r2_test_threshold):
                self._save_block(i, self.blocks[i], model,
                                 r2_train, r2_test, best_features)

        # Remove duplicates
        top_best_features = list(dict.fromkeys(top_best_features))

        # Create final DataFrame containing selected features
        df_top_features = self.df[top_best_features[1:]].copy()
        df_top_features.insert(0, self.frequency_name, self.frequency.values)
        df_top_features[self.target_name] = self.target.values

        print("\n✅ Number of selected features:", len(top_best_features))
        print("✅ Best features:", top_best_features)

        return top_best_features, df_top_features

In [ ]:
df = pd.read_csv(r"final_data-sari.csv")
df = df.drop(['Unnamed: 0', 'voltag'], axis=1, errors='ignore')

df 

,frequency,199.9218775396917,200.0680543122186,200.21422197130983,200.36038051299835,200.50652993331707,200.65267022829892,200.79880139397682,200.94492342638372,201.09103632155256,...,1072.0034971234381,1072.0621337630776,1072.1207438589693,1072.179327408114,1072.237884407512,1072.2964148541635,1072.3549187450692,1072.4133960772294,1072.4718468476444,Max_Oxygen_LifeTime
0,4.0,8.610698,15.138163,0.000000,0.277764,7.499640,19.304629,20.554569,0.000000,26.248741,...,0.00000,16.443610,15.013731,0.000000,0.000000,30.742401,0.000000,28.955052,1.429879,0.000002
1,4.0,12.529087,0.000000,26.037008,36.216891,15.661358,22.317436,27.407377,0.000000,11.746019,...,0.00000,0.000000,11.085487,0.000000,9.321887,38.547263,0.000000,47.365264,0.000000,0.000005
2,4.0,13.511754,22.519590,0.000000,21.808445,32.001522,38.875923,64.240093,0.000000,53.572919,...,0.00000,39.964174,0.000000,0.000000,0.000000,14.948431,0.000000,0.000000,0.000000,0.000005
3,4.0,13.536441,42.113373,26.772073,27.674503,27.975312,68.584637,68.885447,41.511754,37.902036,...,0.00000,75.102847,0.000000,0.000000,0.000000,63.101876,102.976068,0.000000,0.000000,0.000005
4,4.0,14.936498,28.701507,43.638005,37.780555,33.094595,54.181416,65.310572,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,34.299097,42.591187,0.000000,0.000000,0.000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,20.0,0.000000,0.000000,8.842196,0.000000,0.000000,11.395224,0.000000,0.000000,0.000000,...,0.00000,23.646499,0.000000,28.901276,1.681529,0.000000,21.649683,26.694270,0.000000,0.000004
575,20.0,0.000000,0.000000,4.493884,0.000000,2.930794,0.000000,0.000000,2.149249,0.000000,...,0.00000,9.783073,0.000000,4.396887,0.000000,0.000000,18.906613,0.000000,0.000000,0.000003
576,20.0,0.000000,10.679780,12.312358,0.000000,14.285056,0.000000,0.000000,12.856551,0.000000,...,5.74044,12.743778,23.076571,6.773720,0.000000,0.000000,57.748831,0.000000,6.544102,0.000003
577,20.0,0.000000,0.000000,19.268265,0.000000,12.820223,10.165148,0.000000,0.000000,1.972342,...,16.13214,0.000000,13.955581,9.346399,0.000000,0.000000,7.553939,0.000000,0.000000,0.000003


In [ ]:
selector = SpectralFeatureSelector(
    df_start=df,
    target_name="Max_Oxygen_LifeTime",
    frequency_name="frequency",
    block_size=10,
    is_norm=False,
    best_r2_train=0.70,
    best_r2_test=0.58,
    xgb_params = {
            "n_estimators": 150,
            "learning_rate": 0.03,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
            "random_state": 4,
            "min_child_weight": 0.4,
            "n_jobs": -1
        },
    best_features_selected=4,
    # num_blocks=408, 
    save_blocks=True,                    
    save_dir=r".\log\step1",        
    save_r2_train_threshold=0.0,       
    save_r2_test_threshold=0.0
)

top_features, df_final = selector.run()

Block 0
✅ RMSE on Test Set: 0.0615
✅ R² Score on Train Set: 0.8333
✅ R² Score on Test Set: 0.5779
✅ Cross-validation R² scores: [0.5779 0.3515 0.4487 0.316  0.5309]
✅ Mean CV R²: 0.4450 ± 0.1004

Best features based on the XGBoost model itself:
['frequency', '200.36038051299835', '201.23714007551627', '200.79880139397682', '200.65267022829892', '200.50652993331707', '200.94492342638372', '200.0680543122186', '199.9218775396917', '200.21422197130983']
✅ Block 0 saved to D:\دفاع و مقاله\mahmoudi data\saeed mahmoodi\article normaling\moin\sari\filter\final\log\step1\block_0
Block 1
✅ RMSE on Test Set: 0.0679
✅ R² Score on Train Set: 0.8519
✅ R² Score on Test Set: 0.4850
✅ Cross-validation R² scores: [0.485  0.3163 0.4252 0.1775 0.4639]
✅ Mean CV R²: 0.3736 ± 0.1140

Best features based on the XGBoost model itself:
['frequency', '202.40564054779216', '201.38323468430772', '201.96752158841008', '202.55166185239142', '202.11357041183444', '201.82146359997816', '201.52932014395992', '201.6753

In [4]:
df_final

,frequency,206.0534097757587,206.19920036834083,206.49075370938021,208.53058506489424,207.51089762316474,207.94796250064184,227.52736891479316,228.1048056598934,228.82637928221195,...,1048.988868493698,1048.716318608071,1048.920769192638,1050.6156092509214,1051.0199989101457,1050.6830713546747,1070.469637464074,1070.3509066303568,1070.528963188294,Max_Oxygen_LifeTime
0,0.2,6.527465,0.000000,0.000000,5.833053,4.860878,0.000000,9.443991,1.249940,15.554809,...,14.477526,10.902828,0.000000,17.873489,25.022885,0.000000,26.274029,43.968783,25.916559,0.459737
1,0.2,0.000000,14.095222,1.566136,10.571417,0.000000,0.000000,14.682523,0.000000,0.000000,...,0.503886,17.132117,0.000000,0.000000,15.872402,0.000000,7.558287,37.287548,16.628231,0.971616
2,0.2,2.133435,20.623203,21.808445,25.127121,26.312363,0.000000,29.393991,0.000000,16.119285,...,0.000000,22.880252,0.000000,11.592661,54.607535,0.000000,11.592661,73.216807,8.541961,0.911778
3,0.2,0.000000,46.023901,14.739681,72.194354,72.495164,0.000000,13.837251,53.544146,51.739287,...,0.000000,44.519729,0.000000,2.322768,38.712808,0.000000,0.000000,33.293015,4.645537,0.939168
4,0.2,0.000000,0.000000,31.923105,33.973212,54.181416,32.215977,22.551184,0.000000,32.215977,...,13.191960,55.029321,0.000000,15.076526,121.742949,0.000000,22.991702,57.667713,0.753826,0.841546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,1.0,0.186807,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.725050,...,0.000000,0.000000,19.547772,18.707008,0.315287,0.000000,0.000000,0.420382,28.270703,0.738645
575,1.0,6.512875,0.000000,0.000000,0.000000,9.378540,0.000000,0.000000,0.000000,0.000000,...,0.000000,9.673151,4.946498,16.378403,0.000000,0.000000,0.000000,15.499026,49.355053,0.683652
576,1.0,3.741324,0.000000,3.265156,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,18.713836,0.000000,32.031658,11.595690,6.888529,17.795366,0.000000,0.000000,9.299514,0.652843
577,1.0,1.517186,0.000000,15.930455,1.365468,0.910312,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,40.330350,17.540501,0.000000,0.000000,8.194103,22.405750,6.529676,0.622883


In [5]:
df4 = df_final.copy()

In [ ]:
df4.to_csv(r".\log\step1.csv")

In [ ]:
df4 = df_final.copy()
selector = SpectralFeatureSelector(
    df_start=df4,
    target_name="Max_Oxygen_LifeTime",
    frequency_name="frequency",
    block_size=10,
    is_norm=True,
    best_r2_train=0.70,
    best_r2_test=0.60,
    best_features_selected=4,
    # num_blocks=518,
    save_blocks=True,                    
    save_dir=r"D:\دفاع و مقاله\mahmoudi data\saeed mahmoodi\article normaling\moin\sari\filter\final\log\step2",        
    save_r2_train_threshold=0.0,        
    save_r2_test_threshold=0.0
)

top_features, df5 = selector.run()

Block 0
✅ RMSE on Test Set: 0.0593
✅ R² Score on Train Set: 0.8564
✅ R² Score on Test Set: 0.6077
✅ Cross-validation R² scores: [0.6077 0.4863 0.5693 0.4833 0.523 ]
✅ Mean CV R²: 0.5339 ± 0.0483

Best features based on the XGBoost model itself:
['frequency', '355.26607692224155', '227.52736891479316', '228.1048056598934', '206.0534097757587', '228.82637928221195', '207.51089762316474', '208.53058506489424', '206.49075370938021', '206.19920036834083']
✅ Block 0 saved to D:\دفاع و مقاله\mahmoudi data\saeed mahmoodi\article normaling\moin\sari\filter\final\log\step2\block_0
Block 1
✅ RMSE on Test Set: 0.0619
✅ R² Score on Train Set: 0.8549
✅ R² Score on Test Set: 0.5719
✅ Cross-validation R² scores: [0.5719 0.47   0.48   0.4873 0.5105]
✅ Mean CV R²: 0.5039 ± 0.0365

Best features based on the XGBoost model itself:
['frequency', '416.6563392404776', '354.9987170750739', '416.02181011570315', '416.27566784239156', '478.84729931065425', '356.86910080920103', '354.8650169098851', '355.8006345

In [8]:
df6 = df5.copy()

In [ ]:
df6.to_csv(r".\log\step2.csv")

In [10]:
df6

,frequency,355.26607692224155,227.52736891479316,228.1048056598934,508.4579820682637,624.7739429956808,509.2412520609261,746.6584998199332,738.8342325716347,744.5957020753282,...,854.7166023522125,858.2383815524672,851.444148653565,887.258212048309,912.2418726871764,911.7194528327119,1047.003667471031,1047.5534428923963,1000.9177192045671,Max_Oxygen_LifeTime
0,0.2,15.693691,9.443991,1.249940,0.000000,34.094721,0.000000,6.857589,167.325163,5.486071,...,18.577831,0.000000,0.000000,0.000000,386.246099,0.000000,15.371201,12.332707,5.183312,0.459737
1,0.2,0.000000,14.682523,0.000000,0.000000,103.661344,0.000000,31.635450,318.990789,0.000000,...,87.348993,0.000000,43.410868,0.000000,756.836451,1.259714,29.981204,34.012291,0.000000,0.971616
2,0.2,6.163256,29.393991,0.000000,46.937403,237.872811,100.458785,12.768803,501.175525,0.000000,...,58.736495,0.000000,120.452377,0.000000,1054.932155,15.863641,0.000000,12.812941,0.000000,0.911778
3,0.2,0.000000,13.837251,53.544146,0.000000,189.468595,114.004574,47.529882,645.434196,0.000000,...,120.985155,109.912853,163.654026,0.000000,1481.539146,25.550453,54.197931,33.293015,0.000000,0.939168
4,0.2,0.000000,22.551184,0.000000,8.134488,158.228915,106.273152,34.180944,740.674757,0.000000,...,67.573096,0.000000,180.896071,0.000000,1739.831124,12.438134,41.837360,36.183663,11.307395,0.841546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,1.0,11.021610,0.000000,0.000000,0.000000,15.287786,98.902616,107.145214,952.543208,191.080931,...,0.000000,0.000000,348.778338,68.356739,2267.961964,36.573251,23.226117,12.085988,2.627389,0.738645
575,1.0,16.868346,0.000000,0.000000,39.158921,0.000000,189.268120,92.778647,948.403947,120.379463,...,62.517511,112.731044,259.713704,122.042163,2514.359653,26.381320,0.000000,18.796691,15.718870,0.683652
576,1.0,21.019440,0.000000,0.000000,0.000000,0.000000,40.558919,0.000000,948.886917,49.319891,...,0.000000,0.000000,248.336071,81.273623,2804.664412,32.490893,38.805378,32.950128,15.154763,0.652843
577,1.0,32.316066,0.000000,0.000000,0.000000,0.000000,98.823214,144.473708,1428.856588,38.345568,...,0.000000,158.804880,436.519754,0.000000,3477.884529,56.334457,24.070177,22.405750,7.553939,0.622883


In [ ]:
X = df6.drop('Max_Oxygen_LifeTime', axis=1).values
y = df6['Max_Oxygen_LifeTime'].values.reshape(-1, 1)

# Standardize features
scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=4)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
# -----------------------------
# 2. Define the model
# -----------------------------
model = XGBRegressor(
    n_estimators=200,  
    learning_rate=0.1,  
    max_depth=3,           # Same 9 as yours
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=4,
    min_child_weight=0.4,  # Same 0.4 as yours
    n_jobs=-1
)

# -----------------------------
# 3. Train the model on the data
# -----------------------------
model.fit(X_train, y_train.ravel())


# -----------------------------
# 4. Evaluate on Train/Test
# -----------------------------
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)


rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"✅ RMSE on Test Set: {rmse_test:.4f}")
print(f"✅ R² Score on Train Set: {r2_train:.4f}")
print(f"✅ R² Score on Test Set: {r2_test:.4f}")


# -----------------------------
# 5. Cross-Validation with ShuffleSplit
# -----------------------------
cv = ShuffleSplit(n_splits=5, test_size=0.1, random_state=4)
cv_scores = cross_val_score(model, X, y.ravel(), cv=cv, scoring='r2')

print("✅ Cross-validation R² scores:", np.round(cv_scores, 4))
print(f"✅ Mean CV R²: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

importance = pd.DataFrame({
    "Feature": df6.drop(columns="Max_Oxygen_LifeTime").columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

✅ RMSE on Test Set: 0.0488
✅ R² Score on Train Set: 0.9180
✅ R² Score on Test Set: 0.7335
✅ Cross-validation R² scores: [0.7335 0.363  0.4319 0.281  0.4776]
✅ Mean CV R²: 0.4574 ± 0.1532
               Feature  Importance
0            frequency    0.098659
31    887.258212048309    0.063242
28   854.7166023522125    0.047727
24   805.1203002700619    0.045244
11   751.4481941798015    0.034560
7    746.6584998199332    0.032930
30    851.444148653565    0.032124
9    744.5957020753282    0.030991
32   912.2418726871764    0.028515
16   793.3507163270924    0.027174
2   227.52736891479316    0.026813
1   355.26607692224155    0.026590
27   827.6803530142745    0.026104
19   796.7395251048796    0.025132
5    624.7739429956808    0.024642
25   826.6865440727449    0.024118
36  1000.9177192045671    0.023914
21   800.2124522147473    0.023679
35  1047.5534428923963    0.022727
3    228.1048056598934    0.022566
33   911.7194528327119    0.021865
15   762.0475520729605    0.021858
8    738

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib  # <-- Added
import os      # <-- Added

# (Here we assume df5 is already created)
# X = df5.drop('Max_Oxygen_LifeTime', axis=1).values
# y = df5['Max_Oxygen_LifeTime'].values.reshape(-1, 1)

# Standardize features
scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=4)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# 2. Define the model (Exactly your code - unchanged)
# -----------------------------

model = XGBRegressor(
    n_estimators=200,  
    learning_rate=0.1,  
    max_depth=3,           # Same 9 as yours
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=4,
    min_child_weight=0.4,  # Same 0.4 as yours
    n_jobs=-1
)
# -----------------------------
# 3. Train the model on the data
# -----------------------------
model.fit(X_train, y_train.ravel())

# -----------------------------
# 4. Evaluate on Train/Test
# -----------------------------
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"✅ RMSE on Test Set: {rmse_test:.4f}")
print(f"✅ R² Score on Train Set: {r2_train:.4f}")
print(f"✅ R² Score on Test Set: {r2_test:.4f}")

# -----------------------------
# 5. Cross-Validation with ShuffleSplit
# -----------------------------
cv = ShuffleSplit(n_splits=5, test_size=0.1, random_state=4)
cv_scores = cross_val_score(model, X, y.ravel(), cv=cv, scoring='r2')

print("✅ Cross-validation R² scores:", np.round(cv_scores, 4))
print(f"✅ Mean CV R²: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

importance = pd.DataFrame({
    "Feature": df5.drop(columns="Max_Oxygen_LifeTime").columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

# ======================== New Section: Saving ========================
print("\n⏳ Saving files...")

# Create directory to keep things organized
os.makedirs("saved_models", exist_ok=True)

# 1. Save XGBoost model (using the library's standard method)
model.save_model("saved_models/xgb_model.json")

# 2. Save scaler (for standardizing future new data)
joblib.dump(scaler, "saved_models/scaler.pkl")

# 3. Save main dataframe (optional, for backup)
df5.to_csv("saved_models/df5_backup.csv", index=False)

# 4. (Optional) For safety, save feature names as well to prevent errors during prediction
feature_names = df5.drop(columns="Max_Oxygen_LifeTime").columns.tolist()
joblib.dump(feature_names, "saved_models/feature_names.pkl")

print("✅ Saving completed successfully.")
print(f"📁 The following files were created in the 'saved_models' folder:")
print("    - xgb_model.json   (The model itself)")
print("    - scaler.pkl       (Standardization scaler)")
print("    - df5_backup.csv   (Original data)")
print("    - feature_names.pkl (Column names)")

✅ RMSE on Test Set: 0.0488
✅ R² Score on Train Set: 0.9180
✅ R² Score on Test Set: 0.7335
✅ Cross-validation R² scores: [0.7335 0.363  0.4319 0.281  0.4776]
✅ Mean CV R²: 0.4574 ± 0.1532
               Feature  Importance
0            frequency    0.098659
31    887.258212048309    0.063242
28   854.7166023522125    0.047727
24   805.1203002700619    0.045244
11   751.4481941798015    0.034560
7    746.6584998199332    0.032930
30    851.444148653565    0.032124
9    744.5957020753282    0.030991
32   912.2418726871764    0.028515
16   793.3507163270924    0.027174
2   227.52736891479316    0.026813
1   355.26607692224155    0.026590
27   827.6803530142745    0.026104
19   796.7395251048796    0.025132
5    624.7739429956808    0.024642
25   826.6865440727449    0.024118
36  1000.9177192045671    0.023914
21   800.2124522147473    0.023679
35  1047.5534428923963    0.022727
3    228.1048056598934    0.022566
33   911.7194528327119    0.021865
15   762.0475520729605    0.021858
8    738

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

print("⏳ Loading saved files...")

# ==========================================
# 1. Load the saved model, scaler, and dataframe
# ==========================================

# Load XGBoost model
model = xgb.XGBRegressor()
model.load_model("saved_models/xgb_model.json")

# Load scaler (to re-standardize data)
scaler = joblib.load("saved_models/scaler.pkl")

# Load main dataframe (the same df5 you had during training)
df5 = pd.read_csv("saved_models/df5_backup.csv")

print("✅ Files loaded successfully.")

# ==========================================
# 2. Re-prepare data (exactly like the previous code)
# ==========================================

X = df5.drop('Max_Oxygen_LifeTime', axis=1).values
y = df5['Max_Oxygen_LifeTime'].values.reshape(-1, 1)

# Split data into training and testing with the same seed (random_state=4)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=4
)

# Standardize features with the scaler previously fitted on X_train
# (Note: at this stage we only transform, not fit_transform)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 3. Predict and evaluate on Train and Test data
# ==========================================

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate metrics
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print("\n📊 **Evaluation results of the loaded model on df5:**")
print("=========================================")
print(f"✅ RMSE on Test Set : {rmse_test:.4f}")
print(f"✅ R² Score on Train : {r2_train:.4f}")
print(f"✅ R² Score on Test  : {r2_test:.4f}")

# ==========================================
# 4. Display feature importance
# ==========================================

importance = pd.DataFrame({
    "Feature": df5.drop(columns="Max_Oxygen_LifeTime").columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n📌 **Feature Importances:**")
print(importance.to_string(index=False))

⏳ در حال بارگذاری فایل‌های ذخیره‌شده...
✅ فایل‌ها با موفقیت بارگذاری شدند.

📊 **نتایج ارزیابی مدل بارگذاری‌شده روی df5:**
✅ RMSE on Test Set : 0.0488
✅ R² Score on Train : 0.9180
✅ R² Score on Test  : 0.7335

📌 **اهمیت ویژگی‌ها (Feature Importances):**
           Feature  Importance
         frequency    0.098659
  887.258212048309    0.063242
 854.7166023522125    0.047727
 805.1203002700619    0.045244
 751.4481941798015    0.034560
 746.6584998199332    0.032930
  851.444148653565    0.032124
 744.5957020753282    0.030991
 912.2418726871764    0.028515
 793.3507163270924    0.027174
227.52736891479316    0.026813
355.26607692224155    0.026590
 827.6803530142745    0.026104
 796.7395251048796    0.025132
 624.7739429956808    0.024642
 826.6865440727449    0.024118
1000.9177192045671    0.023914
 800.2124522147473    0.023679
1047.5534428923963    0.022727
 228.1048056598934    0.022566
 911.7194528327119    0.021865
 762.0475520729605    0.021858
 738.8342325716347    0.021514
 77